<a href="https://colab.research.google.com/github/Sizan99/ml-pipeline/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sizan99/ml-pipeline/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Method:** Random Forest Classifier.
**Why:** Because this is a ranking task ("which first?"), I need probability scores, not just binary labels. Random Forest handles non-linear interactions well (like the relationship between word count and search volume) and gives clear feature importances to read errors from without needing massive data scaling.

In [1]:
# Verified in Split Design Below

## 2. Split design

**Split Design:** An 80/20 random train-test split on the March 2026 data.
**Why:** Since I am trying to predict our baseline rule (`impressions > 1000` & `ctr < 0.01`), a simple 80/20 split is an honest way to test if the model generalizes using only content features (`word_count`, `search_volume`) without leaking target data.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from google.colab import userdata

# 1. LOAD DATASET
hf_token = userdata.get('HF_TOKEN')
print("Loading fact and dim tables from Hugging Face...")
url_fact = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
url_dim = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

df_daily = pd.read_parquet(url_fact, storage_options={"token": hf_token})
df_dim = pd.read_parquet(url_dim, storage_options={"token": hf_token})

# 2. AGGREGATE TO MONTHLY TARGET
df_monthly = df_daily.groupby('content_hash_id').agg(
    impressions=('gsc_impressions', 'sum'),
    clicks=('gsc_clicks', 'sum')
).reset_index()
df_monthly['ctr'] = (df_monthly['clicks'] / df_monthly['impressions']).fillna(0)

# Create the Ground Truth Label based on the ML-07 Baseline logic
df_monthly['is_failing_snippet'] = ((df_monthly['impressions'] >= 1000) & (df_monthly['ctr'] < 0.01)).astype(int)

# 3. MERGE FEATURES & CLEAN
df = df_monthly.merge(df_dim[['content_hash_id', 'word_count', 'search_volume']], on='content_hash_id', how='inner')
df = df.dropna(subset=['word_count', 'search_volume'])

# 4. SPLIT 80/20
X = df[['word_count', 'search_volume']]
y = df['is_failing_snippet']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Data prepared! Train size: {len(X_train):,}, Test size: {len(X_test):,}")

Loading fact and dim tables from Hugging Face...
Data prepared! Train size: 137,680, Test size: 34,421


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
from sklearn.ensemble import RandomForestClassifier
import numpy as np

# Train Model
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

# Predict Probabilities on Test Set
test_df = df.loc[X_test.index].copy()
test_df['rf_prob'] = rf.predict_proba(X_test)[:, 1]

# Re-calculate Baseline Score strictly on Test Set
test_df['baseline_score'] = test_df['impressions'] * (1.0 - test_df['ctr'])

# Evaluate at Precision@50
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

k = 50
p_baseline = precision_at_k(test_df['baseline_score'], test_df['is_failing_snippet'], k)
p_model = precision_at_k(test_df['rf_prob'], test_df['is_failing_snippet'], k)
base_rate = test_df['is_failing_snippet'].mean()

print(f"--- Comparison Table at K={k} ---")
print(f"Base Rate (Random Guessing): {base_rate:.4f}")
print(f"Baseline Rule Precision@{k}:   {p_baseline:.4f}")
print(f"RF Model Precision@{k}:        {p_model:.4f}")


--- Comparison Table at K=50 ---
Base Rate (Random Guessing): 0.2083
Baseline Rule Precision@50:   0.9400
RF Model Precision@50:        0.5000


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [5]:
# 1. Feature Importances
importances = rf.feature_importances_
print("--- Feature Importances ---")
for feature, imp in zip(X.columns, importances):
    print(f"{feature}: {imp:.4f}")

# Find the top feature
top_idx = importances.argmax()
top_feature = X.columns[top_idx]
top_importance = importances[top_idx]

# 2. Look at 3 False Positives (Model predicted snippet failure, but it wasn't)
test_df_sorted = test_df.sort_values(by='rf_prob', ascending=False)
false_positives = test_df_sorted[test_df_sorted['is_failing_snippet'] == 0].head(3)

print("\n--- 3 False Positives Analysis ---")
for i, row in enumerate(false_positives.itertuples(), 1):
    print(f"{i}. Hash: {row.content_hash_id[:8]}... | Word Count: {row.word_count:,.0f} | Search Vol: {row.search_volume:,.0f} | Impressions: {row.impressions:,.0f}")

# Dynamic Error Analysis output
print(f"\nError Analysis: The model relies almost entirely on {top_feature} ({top_importance:.2f} importance). It assumes that massive values in {top_feature} automatically mean high impressions and a failing snippet. However, as seen in the false positives, pages with extreme {top_feature} often rank poorly and get low impressions, causing the ML model to incorrectly flag them. Our simple Baseline Rule ({p_baseline:.2f}) easily beat the ML Model ({p_model:.2f}) because the rule used actual performance metrics instead of trying to guess!")


--- Feature Importances ---
word_count: 0.9454
search_volume: 0.0546

--- 3 False Positives Analysis ---
1. Hash: content_... | Word Count: 8,507 | Search Vol: 0 | Impressions: 855
2. Hash: content_... | Word Count: 8,558 | Search Vol: 0 | Impressions: 492
3. Hash: content_... | Word Count: 8,368 | Search Vol: 0 | Impressions: 772

Error Analysis: The model relies almost entirely on word_count (0.95 importance). It assumes that massive values in word_count automatically mean high impressions and a failing snippet. However, as seen in the false positives, pages with extreme word_count often rank poorly and get low impressions, causing the ML model to incorrectly flag them. Our simple Baseline Rule (0.94) easily beat the ML Model (0.50) because the rule used actual performance metrics instead of trying to guess!


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.